# Quick start, CPU: a three-link arm in the robosuite `Lift` scene (MuJoCo simulator)

The shortest path from a robot description to a running batched environment: a config written with the builder, `.build()`, `rollout`.
robosuite only supplies the assets (arena, cube, placement rule); our MuJoCo simulation steps everything.

Needs the `robosuite` extra: `pip install -e ".[env,robosuite]"`. Longer tours: `scene_walkthrough.ipynb` (control modes, objectives,
rendering), `scene_viz.ipynb` (videos), `embodiment_sweep.ipynb` (families of arms), `quickstart_gpu_warp.ipynb` (the same scene on a GPU).

In [1]:
import logging

from robotdesigner.config import BaseEnvConfig, Chain, FixedGains
from robotdesigner.tasks.robosuite import LiftObject, RobosuiteParams   # a task comes from the provider that owns it

import robosuite  # noqa: E402,F401  its import-time warnings about optional extras are harmless
logging.getLogger("robosuite_logs").setLevel(logging.WARNING)   # set after the import: robosuite resets its logger level on import

[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /work/yn030245/venv/robotdesigner/lib/python3.13/site-packages/robosuite/scripts/setup_macros.py (macros.py:55)
[robosuite WARNING] Could not import robosuite_models. Some robots may not be available. If you want to use these robots, please install robosuite_models from source (https://github.com/ARISE-Initiative/robosuite_models) or through pip install. (__init__.py:30)
[robosuite WARNING] Could not load the mink-based whole-body IK. Make sure you install related import properly, otherwise you will not be able to use the default IK controller setting for GR1 robot. (__init__.py:40)


## The config, with the builder

`BaseEnvConfig()` is a complete, runnable default (a one-link arm doing the placeholder `empty/idle` task on the CPU MuJoCo simulator).
`.robot()`, `.control()`, `.task()` and `.sim()` each return a **new** config that changes only what the call writes, so the chain below
reads as "this arm, this control law, this task, this many envs". Nothing is mutated, so `lift` can serve as the base of any variant.

In [2]:
lift = (
    BaseEnvConfig()
    .robot(Chain(n=3, axes="zyy", total_length=0.9, total_mass=3.0))          # a yaw joint, then two pitch joints; the rest is the base's
    .control(mode="joint_pos_abs", gains=FixedGains(kp=60.0, kd=6.0))
    .task(params=RobosuiteParams(name="Lift", env_kwargs={}, objective=LiftObject("cube", height=0.84),
                                 reset_noise=0.0, terminate_on_success=False),
          max_episode_steps=50)                                                # the mount is the provider's default for Lift
    .sim(num_envs=4)
)

`resolve()` returns the frozen, validated `EnvConfig` with every value written out (the mount included) and its `env_hash`.
`build()` is `make_env(resolve())`: the one step from config to environment.

In [3]:
cfg = lift.resolve()
print("task:", cfg.task.id, "| mount:", cfg.task.mount.pos, "| env_hash:", cfg.env_hash)

env = lift.build()
print(f"robot_hash={env.robot_hash} env_hash={env.env_hash}")
print("batch_size:", env.batch_size, "| action:", env.action_space)

task: robosuite/Lift | mount: (-0.56, 0.0, 0.912) | env_hash: 61553b4b862d
robot_hash=287b31f43ec4 env_hash=61553b4b862d
batch_size: torch.Size([4]) | action: torch.Size([3])


## Roll out a random policy

The env never autoresets and the episode is truncated after `max_episode_steps`, so `rollout(50)` stops there.

In [4]:
traj = env.rollout(50)
print("rollout batch shape:", tuple(traj.shape))
print("task observations:", sorted(traj["next", "task"].keys()))
print("final cube z in the robot's base frame (m):", traj["next", "task", "cube_pos"][:, -1, 2].tolist())

rollout batch shape: (4, 50)
task observations: ['cube_pos', 'cube_quat', 'ee_pos']
final cube z in the robot's base frame (m): [-0.0906774029135704, -0.09090223163366318, -0.09090223163366318, -0.09090223163366318]


## A variant is another builder call

A longer arm with five joints and more envs: the task, control and simulator carry over from `lift`, and `lift` itself is unchanged.

In [5]:
five = lift.robot(Chain(n=5, axes="zyyyy")).sim(num_envs=8)       # total length and mass carry over
print("lift:", lift.resolve().robot.n, "joints | five:", five.resolve().robot.n, "joints, hash", five.resolve().env_hash)

env5 = five.build()
print(tuple(env5.rollout(10).shape), "| action:", env5.action_space)
env5.close()

lift: 3 joints | five: 5 joints, hash 331d26f1bb44
(8, 10) | action: torch.Size([5])


In [6]:
env.close()